### Burgers' equation
$\frac{\partial u}{\partial t}+uu_x-\frac{0.01}{\pi}u_{xx}=0$, $x\in[-1,1]$, $t\in[0,1]$

$u(0,x)=-sin(\pi x)$

$u(t,-1)=u(t,1)=0$

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import mlp
import utils
import matplotlib.pyplot as plt

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print("operation mode: ", device)

In [ ]:
nt = 100
nx = 64
t = torch.linspace(0,1,nt)
x = torch.linspace(-1,1,nx)
cdata = utils.collocation_points(t,x).to(device)
icdata = utils.collocation_points(t[0],x).to(device)
bcdata = utils.collocation_points(t[[50,99]],x[[0,-1]]).to(device)

uini = -torch.sin(torch.pi*x).reshape(-1,1).to(device)
plt.figure(figsize=(6, 3))
plt.plot(x,uini.cpu().numpy().reshape(-1))
plt.title(r'$u(0,x)=-sin(\pi x)$')
plt.show()

In [ ]:
# nn
num_hidden = 4
num_nodes = 32

layer_list = [2]+num_hidden*[num_nodes]+[1]
model = mlp.pinn(layer_list).to(device)

In [ ]:
ls = 10

cdata_req = cdata.clone()
cdata_req.requires_grad = True

for _ in range(num_epochs):
    optimizer.zero_grad()
    u0pred = model(icdata)
    ubcpred = model(bcdata)
    upred = model(cdata_req)
    loss_col = torch.mean(utils.burgers_equation(upred, cdata_req)**2)
    loss_ini = torch.mean((u0pred-uini)**2)
    loss_bc = torch.mean(ubcpred**2)

    loss = loss_col+loss_ini+loss_bc
    loss.backward()
    optimizer.step()
    if loss.item() < ls:
        ls = loss.item()
        torch.save(model.state_dict(), './params/bg.pt')

print(f'loss: {ls}') 

In [ ]:
# result
nt = 101
nx = 256
t = torch.linspace(0,1,nt)
x = torch.linspace(-1,1,nx)
cdata = utils.collocation_points(t,x).to(device)

model.load_state_dict(torch.load('./param/bg.pt',map_location=device))

with torch.no_grad():
    upred = model(cdata).cpu().numpy()

# reference
usol = np.load('./sols/bgsol.npy') # (101, 256)

In [ ]:
relerr = utils.relative_l2_error(usol,upred.reshape(nt,nx))
print(f'Averaged Relative L2 error: {100*np.mean(relerr):.3f}%')

In [ ]:
fig = plt.figure(figsize=(6, 2))
plt.plot(t,100*relerr,'--',color='black')
plt.xlabel(r'$t$')
plt.ylabel('%')
plt.title(r'Relative $L_2$ error $\frac{\parallel u_{FEM}-u_{PINN}\parallel_2}{\parallel u_{FEM}\parallel_2}$')
plt.show()

In [ ]:
fig = plt.figure(figsize=(8, 5))
gs = fig.add_gridspec(2, 3)

ax1 = fig.add_subplot(gs[0, :])
im1 = ax1.imshow(upred.reshape(nt,nx).T, interpolation='nearest', cmap='bwr',
            extent=[t.min(), t.max(), x.min(), x.max()],
            origin='lower', aspect='auto')
cbar = plt.colorbar(im1)
cbar.set_ticks([-1, 0, 1])
ax1.set_title(r'$u(t,x)$')
ax1.set_xlabel(r'$t$')
ax1.set_ylabel(r'$x$')

n1, n2, n3 = 25, 50, 99
for i, n in enumerate([n1, n2, n3]):
    ax = fig.add_subplot(gs[1, i])
    ax.plot(x, usol[n], color='blue')
    ax.plot(x, upred[n*nx:(n+1)*nx], 'r--')
    ax.set_title(rf'$t={t[n]:.1f}$')
    ax.set_ylim(-1.1, 1.1)
    ax.set_xlabel(r'$x$')
    ax.legend(labels=[r'$u_{FEM}$',r'$u_{PINN}$'])

plt.tight_layout()
plt.show()